In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
df = pd.read_csv("../data/processed/cases_final.csv")

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df["text_full"])

In [10]:
def retrieve(query, k=1):
    query_vector = vectorizer.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_index = similarity.argsort()[-k:][::-1]
    return df.iloc[top_index]["case_id"].tolist()

In [11]:
eval_df = df[["case_id", "text_full"]].head(5).copy()

eval_df["query_id"] = ["q1", "q2", "q3", "q4", "q5"]

eval_df["query"] = eval_df["text_full"].apply(
    lambda x: " ".join(str(x).split()[:80])
)

eval_df["ground_truth"] = eval_df["case_id"]

eval_df = eval_df[["query_id", "query", "ground_truth"]]

eval_df

,query_id,query,ground_truth
0,q1,penetapan nomor 105/pdt.g/2025/pn sda esi demi...,case_001
1,q2,putusan nomor 148/pdt.g/2025/pn sda demi keadi...,case_002
2,q3,direktori putusan mahkamah agung republik indo...,case_003
3,q4,direktori putusan mahkamah agung pdt.i.c.3 put...,case_004
4,q5,penetapan nomor 213/pdt.g/2026/pn sda demi kea...,case_005


In [12]:
predictions = []

for query in eval_df["query"]:
    pred = retrieve(query, k=1)[0]
    predictions.append(pred)

eval_df["predicted"] = predictions
eval_df

,query_id,query,ground_truth,predicted
0,q1,penetapan nomor 105/pdt.g/2025/pn sda esi demi...,case_001,case_001
1,q2,putusan nomor 148/pdt.g/2025/pn sda demi keadi...,case_002,case_002
2,q3,direktori putusan mahkamah agung republik indo...,case_003,case_003
3,q4,direktori putusan mahkamah agung pdt.i.c.3 put...,case_004,case_004
4,q5,penetapan nomor 213/pdt.g/2026/pn sda demi kea...,case_005,case_005


In [13]:
accuracy = accuracy_score(eval_df["ground_truth"], eval_df["predicted"])
precision = precision_score(eval_df["ground_truth"], eval_df["predicted"], average="macro", zero_division=0)
recall = recall_score(eval_df["ground_truth"], eval_df["predicted"], average="macro", zero_division=0)
f1 = f1_score(eval_df["ground_truth"], eval_df["predicted"], average="macro", zero_division=0)

metrics_df = pd.DataFrame([{
    "model": "TF-IDF + Cosine Similarity",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1
}])

metrics_df

,model,accuracy,precision,recall,f1_score
0,TF-IDF + Cosine Similarity,1.0,1.0,1.0,1.0


In [14]:
eval_df.to_csv("../data/eval/queries_result.csv", index=False)
metrics_df.to_csv("../data/eval/retrieval_metrics.csv", index=False)

print("Evaluasi berhasil disimpan")

Evaluasi berhasil disimpan
